# Chi-square, t, And F Distributions For Inference

**Official MA1001B Alignment:** *4.4 chi-square; 4.5 t; 4.6 F; 4.7 data science links.*


## How To Use This Lesson

This notebook is designed as a guided teaching episode and interactive lab, not a passive code demonstration. To get the most out of this lesson:
1. **Read the conceptual explanations and explicit links** before running any code.
2. **Execute code cells sequentially**, paying attention to inline educational comments.
3. **Pause at the Guided Checkpoint** to discuss with a partner and write your reasoning before checking solutions.
4. **Complete the Independent Practice and Exit Ticket**; written justification is the primary evidence of statistical competence.


## Learning Goals

By the end of this lesson, you will be able to:
- Select the appropriate inferential reference distribution (`t`, `chi-square`, `F`) based on the statistical estimation task.
- Calculate one-sample t-statistics and p-values using degrees of freedom (`df = n - 1`) when population variance is unknown.
- Construct exact parametric t-distribution confidence intervals using SciPy (`stats.t.interval`).
- Interpret degrees of freedom as an adjustment for estimation uncertainty in small samples.


## The Three Explicit Links

In accordance with the MA1001B pedagogical framework, this lesson explicitly connects theory, computation, and action:

- **1. Conceptual Link (What is modeled):** We model standardized inferential statistics under null hypotheses to quantify how surprising an observed sample metric is.
- **2. Computational Link (How Python represents it):** We use SciPy statistical probability functions (`stats.t.sf`, `stats.t.ppf`, `stats.t.interval`) to evaluate critical values and p-values.
- **3. Decision Link (How it guides action):** Using exact inferential distributions prevents false-positive decisions in small samples where Normal approximations fail.


## Decision Scenario

> **The Problem:** A quality analyst must decide whether a process mean differs from a target when the population variance is unknown.


## Conceptual Explanation

Inferential distributions appear when statistics are standardized. The t distribution accounts for uncertainty from estimating the standard deviation. Chi-square distributions arise in variance and categorical-count settings. F distributions arise in variance ratios and ANOVA.


## Mathematical Anchor

For a one-sample mean with unknown variance, t = (xbar - mu0) / (s / sqrt(n)), with n - 1 degrees of freedom.


## Data And Workflow Notes

Uses a simulated process measurement so the role of the reference distribution is clear.


## Practical Python Workflow

The following worked example demonstrates how to implement these statistical concepts in Python to generate evidence for decision making.


### Step 1: Process Measurement & T-Statistic Calculation

We simulate 25 manufacturing process measurements, define a quality target of 9.0, and manually compute the one-sample t-statistic and two-sided p-value.


In [ ]:
# Import required data science and statistical libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set reproducible random seed and visual styling
rng = np.random.default_rng(1001)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

# Simulate n=25 process measurements against a target of 9.0
target = 9.0
measurements = pd.Series(rng.normal(loc=10.0, scale=2.5, size=25), name="process_measurement")

# Calculate sample mean, sample standard deviation, and standardized t-statistic
xbar = measurements.mean()
s = measurements.std(ddof=1)
n = len(measurements)
t_stat = (xbar - target) / (s / np.sqrt(n))
p_value = 2 * stats.t.sf(abs(t_stat), df=n - 1)

pd.Series({
    "sample_mean_xbar": xbar,
    "sample_sd_s": s,
    "t_statistic": t_stat,
    "degrees_of_freedom": n - 1,
    "two_sided_p_value": p_value,
}).round(4)


### Step 2: Reference Guide: When to Use t, Chi-Square, and F

We construct an architectural reference table summarizing the typical data science questions and 95th percentile critical values for key inferential distributions.


In [ ]:
# Reference guide for inferential distributions
reference = pd.DataFrame({
    "distribution": ["Student's t", "Chi-Square (chi2)", "Fisher's F"],
    "primary_application": [
        "Inference for means with unknown population variance",
        "Inference for sample variance & categorical independence",
        "Comparing ratios of variances & ANOVA group comparisons",
    ],
    "example_95th_percentile_val_(df=24)": [
        stats.t.ppf(0.95, df=24),
        stats.chi2.ppf(0.95, df=24),
        stats.f.ppf(0.95, dfn=3, dfd=24),
    ],
})
reference.round(3)


### Step 3: Parametric T-Distribution Confidence Interval

We use SciPy's automated `stats.t.interval` method to calculate the exact 95% confidence interval for the process mean and compare it against the target value.


In [ ]:
# Compute 95% confidence interval using Student's t-distribution
ci_low, ci_high = stats.t.interval(
    confidence=0.95,
    df=n - 1,
    loc=xbar,
    scale=s / np.sqrt(n),
)

pd.Series({
    "target_val": target,
    "sample_mean": xbar,
    "95%_CI_lower_bound": ci_low,
    "95%_CI_upper_bound": ci_high,
    "target_is_within_CI": ci_low <= target <= ci_high
}).round(4)


### Step 4: Visualizing Sample Distribution vs. Target

We plot the histogram of the 25 process measurements, highlighting the target specification line versus the observed sample mean.


In [ ]:
# Plot measurement distribution against quality target
ax = sns.histplot(measurements, bins=10, kde=True, color="darkorange")
ax.axvline(target, color="red", linestyle="--", linewidth=2, label=f"Target Specification ({target})")
ax.axvline(xbar, color="black", linestyle=":", linewidth=2, label=f"Sample Mean ({xbar:.2f})")
ax.set_title("Process Measurements vs. Target Specification (n=25)", fontsize=14, pad=10)
ax.set_xlabel("Measurement Unit", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.legend()
plt.show()


## Guided Checkpoint

> [!IMPORTANT]
> **Pair Discussion & Writing Prompt:**
> Why must we use a Student's t-distribution instead of a standard Normal distribution when evaluating this sample of 25 measurements?

*Write your reasoned response below before continuing:*


## Common Mistakes & Statistical Pitfalls

Avoid these frequent errors when conducting or communicating this analysis:
- **Warning:** Using the standard Normal distribution (z-scores) when population variance is unknown and sample size is small.
- **Warning:** Treating degrees of freedom as a decorative formula parameter rather than an essential adjustment for sample uncertainty.
- **Warning:** Reporting a standalone p-value without stating the point estimate, effect size, and confidence interval.


## Independent Practice

> [!TIP]
> **Your Task:**
> Change the sample size from `n=25` to `n=10` and then `n=100` in the simulation. Observe and explain how the t-distribution critical value and CI width change.

*Use the empty code and markdown cells below to implement your analysis and justify your recommendation.*


In [ ]:
# Write your independent practice code here
# Remember to inspect your outputs and check assumptions


## Decision Interpretation Template

Use this structured format to write your defensible conclusion and recommendation:

1. **The Decision Question:** *State the practical question being answered...*
2. **The Statistical Evidence:** *Summarize key metrics, intervals, p-values, or model comparisons...*
3. **Uncertainty & Limitations:** *Identify what the data cannot prove and what assumptions were made...*
4. **Actionable Recommendation:** *Therefore, I recommend [action] because [justification]...*


## Exit Ticket

> **Reflection:** What specific role does the estimated standard error (`s / sqrt(n)`) play in the denominator of the t-statistic?

*Write your brief conceptual reflection below:*
